# Convert HOT's Freetown trees to shards

Run on a CPU runtime. Set `REPO_SHA` in the next cell, then run the cells in order.

The labels, tile table and image parquets are downloaded from Hugging Face (pinned at `REVISION`) to `/content`; the images are streamed one record batch at a time, so they are never all in memory. Conversion never reads through the Drive mount: only the finished shards are written back to Drive.

By default HOT `train` and `validation` are converted (as `train` and `val`) into `shards/hot`. With `CONVERT_TEST = True` HOT `test` alone is converted into `shards/hot-test`, the held-out benchmark part: the last cell writes `benchmark_part_hot-test.json` beside the shards, to be frozen in the repo.

The repo is public; this notebook still clones it with a read-only GitHub token from Colab Secrets (the key icon), named `GITHUB_TOKEN`, with notebook access on.

With `RELABEL_M` set, the HOT train labels with those exact tags get the new crown side and the shards go to `shards/hot-relabel`; val regions are written exactly as in `shards/hot`, so runs on either source pair on the same val units.

With `RELABELED_BOXES = True` every tree outside a group takes its hand-drawn crown box from `raw/hot-relabeled/hot_relabeled_boxes.csv` on Drive (`cant_tell` and `merged` trees become ignore squares, `not_a_tree` points get nothing), in train and val into `shards/hot-relabeled`, or with `CONVERT_TEST` in test into `shards/hot-test-relabeled`. Groups, regions, chips and masks are as in the plain conversion; the check cell asserts that against `shards/hot` (or `shards/hot-test`).


In [ ]:
# True: HOT test on its own into shards/hot-test (the held-out benchmark part); False: train and val
CONVERT_TEST = False
# exact tag value -> crown side for HOT train labels only, e.g. {1.0: 2.5, 2.0: 3.75}; val keeps its tags
RELABEL_M = {}
assert not (CONVERT_TEST and RELABEL_M), "relabelling changes training labels; test is never relabelled"
# True: every tree outside a group takes its hand-drawn box (train and val, or test with CONVERT_TEST)
RELABELED_BOXES = False
RELABELED_CSV = "raw/hot-relabeled/hot_relabeled_boxes.csv"  # relative to the Drive root
assert not (RELABELED_BOXES and RELABEL_M), "one tree relabelling at a time"
PLAIN = "hot-test" if CONVERT_TEST else "hot"  # the source a relabelled conversion is checked against
SOURCE = f"{PLAIN}-relabeled" if RELABELED_BOXES else ("hot-relabel" if RELABEL_M else PLAIN)
REPO_SHA = ""
RESOLVE_ZIP = "raw/resolve/Ecoregions2017.zip"  # relative to the Drive root
BLOCK_TILES = 16  # tiles per block side: one region per connected run of tiles in a block

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/fair_trees")

assert REPO_SHA, "set REPO_SHA to the pushed commit"
# a read-only GitHub token in Colab Secrets (key icon), named GITHUB_TOKEN
REPO = "github.com/OmdenaAI/HOTOSM-fAIr-Trees.git"
token = userdata.get("GITHUB_TOKEN")
!rm -rf /content/fair_trees
# git clone through subprocess so the token never reaches the cell output
url = f"https://x-access-token:{token}@{REPO}"
clone = subprocess.run(["git", "clone", "-q", url, "/content/fair_trees"])
# checked by hand: a CalledProcessError message would print the URL, token included
if clone.returncode:
    raise RuntimeError(f"git clone failed (exit {clone.returncode}): check the GITHUB_TOKEN secret")
# drop the token from the saved remote so it is not left in .git/config
git = ["git", "-C", "/content/fair_trees"]
subprocess.run([*git, "remote", "set-url", "origin", f"https://{REPO}"], check=True)
subprocess.run([*git, "checkout", "-q", REPO_SHA], check=True)
del token, url

%pip install /content/fair_trees -r /content/fair_trees/experiments/requirements.txt
sys.path.insert(0, "/content/fair_trees")

In [ ]:
import shutil
import zipfile

from huggingface_hub import hf_hub_download

from experiments.label_study.hot import REPO_ID, REVISION, read_hot

FILES = {
    "labels": "stac/project-50661/labels.parquet",
    "tiles": "stac/project-50661/tiles.parquet",
}
IMAGES = {
    "train": ["data/train-00000-of-00002.parquet", "data/train-00001-of-00002.parquet"],
    "validation": ["data/validation-00000-of-00001.parquet"],
    "test": ["data/test-00000-of-00001.parquet"],
}
IMAGES = {split: paths for split, paths in IMAGES.items() if (split == "test") == CONVERT_TEST}


def fetch(path):
    return Path(
        hf_hub_download(REPO_ID, path, repo_type="dataset", revision=REVISION, local_dir="/content/hot_raw")
    )


labels_path, tiles_path = fetch(FILES["labels"]), fetch(FILES["tiles"])
image_files = {split: [fetch(p) for p in paths] for split, paths in IMAGES.items()}
data = read_hot(labels_path, tiles_path)
for split, paths in image_files.items():
    print(split, [f"{p.name}: {p.stat().st_size / 1e6:.0f} MB" for p in paths])

resolve_dir = Path("/content/resolve")
resolve_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(DRIVE / RESOLVE_ZIP) as z:
    z.extractall(resolve_dir)
print(sorted(p.name for p in resolve_dir.rglob("*.shp")))

In [ ]:
import pyarrow.parquet as pq

from experiments.biome import resolve_biome_fn
from experiments.converters.hot import SPLITS, TEST_SPLITS, convert, read_relabeled_boxes


def images(split):
    """Yield (tile_id, image bytes) for every tile of a HOT split, one record batch at a time."""
    for path in image_files[split]:
        for batch in pq.ParquetFile(path).iter_batches(columns=["tile_id", "image"], batch_size=64):
            for row in batch.to_pylist():
                yield row["tile_id"], row["image"]["bytes"]


relabeled_boxes = None
if RELABELED_BOXES:  # copied off the Drive mount first, like every other input
    shutil.copyfile(DRIVE / RELABELED_CSV, "/content/hot_relabeled_boxes.csv")
    relabeled_boxes = read_relabeled_boxes(Path("/content/hot_relabeled_boxes.csv"))
    print(len(relabeled_boxes), "hand-drawn records")
shapefile = next(resolve_dir.rglob("*.shp"))
converted = Path("/content/converted") / SOURCE
# scratch output on the Colab VM only; convert refuses a non-empty output directory
shutil.rmtree(converted, ignore_errors=True)
stats = convert(
    data,
    images,
    converted,
    biome_fn=resolve_biome_fn(shapefile),
    block_tiles=BLOCK_TILES,
    splits=TEST_SPLITS if CONVERT_TEST else SPLITS,
    relabel_m=RELABEL_M or None,
    relabeled_boxes=relabeled_boxes,
)
print(stats)

In [ ]:
import json

# a relabelled conversion: everything but the tree labels must match the plain conversion of the same split
TREE_LABEL_STATS = {
    "relabelled",
    "relabel_m",
    "relabeled_boxes",
    "ignored_relabeled",
    "dropped_not_a_tree",
    # with relabeled boxes: unknown tags get boxes, not_a_tree squares stop masking, label-free tiles change
    "boxes",
    "ignored_unknown_diameter",
    "masked_other_split",
    "label_free_tile_share",
}
plain_stats = DRIVE / "shards" / PLAIN / "stats.json"
if SOURCE != PLAIN:
    assert plain_stats.exists(), f"convert {PLAIN} first: the check needs its stats.json"
    plain = json.loads(plain_stats.read_text())
    allowed = TREE_LABEL_STATS if RELABELED_BOXES else {"relabelled", "relabel_m"}
    differ = {k: (plain.get(k), v) for k, v in stats.items() if k not in allowed and plain.get(k) != v}
    assert not differ, f"differs from shards/{PLAIN} beyond the tree labels: {differ}"
    for k in sorted(TREE_LABEL_STATS & set(stats)):
        print(f"{k}: {plain.get(k)} -> {stats[k]}")
    print(f"all other stats match shards/{PLAIN}")

In [ ]:
from experiments.shards import manifest_hash, write_shards

shard_dir = DRIVE / "shards" / SOURCE
if (shard_dir / "manifest.json").exists():
    # keep the previous conversion (the runs trained on it record its manifest) instead of writing over it
    old = shard_dir.with_name(f"{SOURCE}-{manifest_hash(shard_dir / 'manifest.json')[:8]}")
    shard_dir.rename(old)
    print("previous shards moved to", old)
write_shards(converted, shard_dir, source=SOURCE, converter_sha=REPO_SHA)
print("manifest sha256:", manifest_hash(shard_dir / "manifest.json"))
shutil.copyfile(converted / "stats.json", shard_dir / "stats.json")

if CONVERT_TEST:  # a HOT test part of the benchmark: send this file to freeze it in the repo
    import datetime
    import json

    from experiments.benchmark import regions_part

    part = regions_part(
        converted.parent,
        SOURCE,
        split="test",
        zooms=[20, 19],
        window_m=60.0,
        frozen_on=datetime.date.today().isoformat(),
    )
    part_path = shard_dir / f"benchmark_part_{SOURCE}.json"
    part_path.write_text(json.dumps(part, indent=2) + "\n")
    print(len(part["unit_ids"]), "test units; written to", part_path)

In [ ]:
import random

from affine import Affine
from IPython.display import Image, display
from PIL import Image as PILImage
from PIL import ImageDraw

from fair_trees.dataset import labels_to_boxes, load_geojson
from fair_trees.georef import tile_bounds_3857
from fair_trees.preprocess import CHIP_SUFFIXES, read_chip

THUMB = 256
chip_paths = sorted(p for p in converted.glob("*/z*/chips/*") if p.suffix.lower() in CHIP_SUFFIXES)
picked = random.Random(0).sample(chip_paths, min(16, len(chip_paths)))
contact = PILImage.new("RGB", (4 * THUMB, 4 * THUMB))
for i, path in enumerate(picked):
    rgb, _, (x, y, z) = read_chip(path, use_sidecar=False)
    minx, miny, maxx, maxy = tile_bounds_3857(x, y, z)
    res = (maxx - minx) / rgb.shape[1]
    transform = Affine(res, 0, minx, 0, -res, maxy)
    region = path.parents[2]
    tile = PILImage.fromarray(rgb[..., :3])
    draw = ImageDraw.Draw(tile)
    # ignore features (unknown-diameter crowns, other-split crowns and groups) in grey,
    # tree labels in red, tree_group labels in green
    layers = (
        ("ignore", 1, (128, 128, 128)),
        ("labels", 1, (255, 0, 0)),
        ("labels", 2, (0, 255, 0)),
    )
    for name, label, colour in layers:
        geojson = load_geojson(region / f"{name}.geojson")
        if name == "ignore":  # ignore features carry no label: read them as trees
            geojson = {"features": [{**f, "properties": {}} for f in geojson["features"]]}
        for x0, y0, x1, y1 in labels_to_boxes(geojson, transform, label):
            draw.rectangle((x0, y0, x1, y1), outline=colour, width=2)
    contact.paste(tile.resize((THUMB, THUMB)), ((i % 4) * THUMB, (i // 4) * THUMB))
sheet = shard_dir / "qa-contact-sheet.png"
contact.save(sheet)
display(Image(filename=str(sheet)))

In [ ]:
import time

largest = max((p for p in shard_dir.iterdir() if p.suffix == ".tar"), key=lambda p: p.stat().st_size)
start = time.monotonic()
shutil.copyfile(largest, Path("/content") / largest.name)
elapsed = time.monotonic() - start
print(f"{largest.name}: {largest.stat().st_size / 1e6 / elapsed:.1f} MB/s")
# Record this in experiments/DATA_SOURCES.md; restaging must take under 10 minutes.

In [ ]:
drive.flush_and_unmount()